# ANGSI JR. V10.2.1 — Jupyter launcher
Upload `ANGSI_JR_V10_2_1_WORKING_AI_CHAT.zip` to `/data/epic20/APGCE/apgce_team_13/`, then run the next cells sequentially.


## 1. Extract ZIP


In [ ]:
from pathlib import Path
import zipfile
ROOT = Path('/data/epic20/APGCE/apgce_team_13')
archive = ROOT / 'ANGSI_JR_V10_2_1_WORKING_AI_CHAT.zip'
assert archive.exists(), f'Missing uploaded ZIP: {archive}'
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, 'Corrupt ZIP'
    z.extractall(ROOT)
APP_DIR = ROOT / 'ANGSI_JR_V10_2_1_WORKING_AI_CHAT'
APP = APP_DIR / 'angsi_jr_v10.py'
print('App:', APP.exists(), APP)
print('Live AI module:', (APP_DIR/'ai_chat.py').exists())
print('Volumetrics module:', (APP_DIR/'v10_2_reservoir_engine.py').exists())

## 2. Install requirements


In [ ]:
%pip install -r /data/epic20/APGCE/apgce_team_13/ANGSI_JR_V10_2_1_WORKING_AI_CHAT/requirements.txt

## 3. Start Streamlit · 8520


In [ ]:
import os, sys, subprocess, py_compile
from pathlib import Path
APP_DIR = Path('/data/epic20/APGCE/apgce_team_13/ANGSI_JR_V10_2_1_WORKING_AI_CHAT')
APP = APP_DIR / 'angsi_jr_v10.py'
LOG = APP_DIR / 'server_v1021.log'
PORT = 8520
assert APP.is_file(), f'App not found: {APP}'
py_compile.compile(str(APP), doraise=True)
env = os.environ.copy()
env['ANGSI_DATA_ROOT'] = '/data/epic20/APGCE/input_data/challenge3'
env['LLM_BASE_URL'] = 'http://172.28.34.1:8000/v1'
env['LLM_MODEL'] = 'google/gemma-4-26B-A4B-it'
env['LLM_API_KEY'] = 'token-not-needed'
env['NO_PROXY'] = '127.0.0.1,localhost,172.28.34.1'
with LOG.open('a') as out:
    process_v1021 = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', str(APP),
         '--server.address=127.0.0.1', f'--server.port={PORT}',
         '--server.headless=true', '--server.fileWatcherType=none'],
        cwd=str(APP_DIR), env=env, stdout=out, stderr=subprocess.STDOUT,
        start_new_session=True)
print('Process:', process_v1021.pid, 'LOG:', LOG)

## 4. Health check


In [ ]:
import time, urllib.request, socket
print('Jupyter kernel host:', socket.gethostname())
for i in range(12):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8520/_stcore/health', timeout=4) as r:
            print('HTTP', r.status, 'response:', r.read().decode())
        break
    except Exception as exc:
        if i == 11:
            print('Health failed:', exc)
            print(LOG.read_text(errors='replace')[-7000:] if LOG.exists() else 'No log')
        else:
            time.sleep(1)

## 5. Browser link


In [ ]:
from IPython.display import display, HTML
from urllib.parse import quote
prefix = '/user/apgce_team_13/'
display(HTML(f'<a target="_blank" href="{prefix}proxy/8520/">OPEN ANGSI JR V10.2.1</a>'))

## 6. Diagnostics


In [ ]:
import os
from pathlib import Path
APP_DIR = Path('/data/epic20/APGCE/apgce_team_13/ANGSI_JR_V10_2_1_WORKING_AI_CHAT')
print((APP_DIR/'server_v1021.log').read_text(errors='replace')[-7000:])

# Organiser server connectivity, from the SAME Python kernel; no data sent:
from urllib.request import urlopen
try:
    print('Model endpoint HTTP:', urlopen('http://172.28.34.1:8000/v1/models', timeout=5).status)
except Exception as exc:
    print(type(exc).__name__, str(exc)[:240])